# Pandas Feature Engineering: Creating, Transforming, and Encoding Features

In [2]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    'customer_id': [101, 102, 103, 104, 105, 106, 107, 108],
    'age': [25, 42, 17, 60, 35, np.nan, 150, 29],
    'monthly_income': [30000, 50000, 18000, 70000, 40000, 0, 55000, 35000],
    'monthly_expenses': [18000, 32000, 12000, 45000, 25000, 10000, 30000, 20000],
    'city': ['Delhi', 'Mumbai', 'Delhi', 'Pune', 'Mumbai', 'Pune', 'Delhi', 'Mumbai'],
    'purchases': [2, 5, 1, 8, 4, 0, 6, 3],
    'customer_type': ['Regular', 'Premium', 'New', 'Premium', 'Regular', 'New', 'Premium', 'Regular']
})
df

,customer_id,age,monthly_income,monthly_expenses,city,purchases,customer_type
0,101,25.0,30000,18000,Delhi,2,Regular
1,102,42.0,50000,32000,Mumbai,5,Premium
2,103,17.0,18000,12000,Delhi,1,New
3,104,60.0,70000,45000,Pune,8,Premium
4,105,35.0,40000,25000,Mumbai,4,Regular
5,106,NaN,0,10000,Pune,0,New
6,107,150.0,55000,30000,Delhi,6,Premium
7,108,29.0,35000,20000,Mumbai,3,Regular


#### Q1. Create Derived Numerical Features
Create a monthly_savings column by subtracting monthly_expenses from monthly_income. Create a savings_rate column representing monthly savings as a percentage of monthly income. Handle zero income safely so the result contains no infinite values.

In [4]:
df['monthly_savings']=df['monthly_income']-df['monthly_expenses']
df['savings_rate']=(df['monthly_savings']/df['monthly_income'])*100
df.loc[df['monthly_income'] == 0, 'savings_rate'] = np.nan


In [5]:
df

,customer_id,age,monthly_income,monthly_expenses,city,purchases,customer_type,monthly_savings,savings_rate
0,101,25.0,30000,18000,Delhi,2,Regular,12000,40.000000
1,102,42.0,50000,32000,Mumbai,5,Premium,18000,36.000000
2,103,17.0,18000,12000,Delhi,1,New,6000,33.333333
3,104,60.0,70000,45000,Pune,8,Premium,25000,35.714286
4,105,35.0,40000,25000,Mumbai,4,Regular,15000,37.500000
5,106,NaN,0,10000,Pune,0,New,-10000,NaN
6,107,150.0,55000,30000,Delhi,6,Premium,25000,45.454545
7,108,29.0,35000,20000,Mumbai,3,Regular,15000,42.857143


#### Q2. Apply Conditional Classification
Create an age_group column with these categories:
- Below 18: Minor
- 18–35: Young Adult
- 36–59: Middle-aged
- 60 and above: Senior
Keep missing and invalid ages, such as 150, separate rather than classifying them as valid ages.

In [11]:
import numpy as np

conditions = [
    df['age'].between(0, 17),
    df['age'].between(18, 35),
    df['age'].between(36, 59),
    df['age'].between(60, 120)
]

choices = [
    'Minor',
    'Young Adult',
    'Middle-aged',
    'Senior'
]

df['age_group'] = np.select(
    conditions,
    choices,
    default='Invalid/Missing'
)
df

,customer_id,age,monthly_income,monthly_expenses,city,purchases,customer_type,monthly_savings,savings_rate,age_group
0,101,25.0,30000,18000,Delhi,2,Regular,12000,40.000000,Young Adult
1,102,42.0,50000,32000,Mumbai,5,Premium,18000,36.000000,Middle-aged
2,103,17.0,18000,12000,Delhi,1,New,6000,33.333333,Minor
3,104,60.0,70000,45000,Pune,8,Premium,25000,35.714286,Senior
4,105,35.0,40000,25000,Mumbai,4,Regular,15000,37.500000,Young Adult
5,106,NaN,0,10000,Pune,0,New,-10000,NaN,Invalid/Missing
6,107,150.0,55000,30000,Delhi,6,Premium,25000,45.454545,Invalid/Missing
7,108,29.0,35000,20000,Mumbai,3,Regular,15000,42.857143,Young Adult


#### Q3. Transform Values Using NumPy
Create an income_level column using np.select():
- Income below 25,000: Low
- Income from 25,000 through 49,999: Medium
- Income of 50,000 or above: High

In [16]:
conditions=[
    df['monthly_income'].between(0,24999),
    df['monthly_income'].between(25000,49999),
    df['monthly_income']>=50000
]

choices=['Low','Medium','High']
df['income_level']=np.select(conditions,choices,default='Uncategorized')

df

,customer_id,age,monthly_income,monthly_expenses,city,purchases,customer_type,monthly_savings,savings_rate,age_group,income_level
0,101,25.0,30000,18000,Delhi,2,Regular,12000,40.000000,Young Adult,Medium
1,102,42.0,50000,32000,Mumbai,5,Premium,18000,36.000000,Middle-aged,High
2,103,17.0,18000,12000,Delhi,1,New,6000,33.333333,Minor,Low
3,104,60.0,70000,45000,Pune,8,Premium,25000,35.714286,Senior,High
4,105,35.0,40000,25000,Mumbai,4,Regular,15000,37.500000,Young Adult,Medium
5,106,NaN,0,10000,Pune,0,New,-10000,NaN,Invalid/Missing,Low
6,107,150.0,55000,30000,Delhi,6,Premium,25000,45.454545,Invalid/Missing,High
7,108,29.0,35000,20000,Mumbai,3,Regular,15000,42.857143,Young Adult,Medium


#### Q4. Normalize a Numerical Feature
Create an income_minmax column by scaling monthly income to the range 0–1 using min-max normalization. Handle the zero-income row correctly. Explain what you would do if every income value were identical.

In [19]:
income_min = df['monthly_income'].min()
income_max = df['monthly_income'].max()

if income_max != income_min:
    df['income_minmax'] = (
        (df['monthly_income'] - income_min) /
        (income_max - income_min)
    )
else:
    df['income_minmax'] = 0.0

df

,customer_id,age,monthly_income,monthly_expenses,city,purchases,customer_type,monthly_savings,savings_rate,age_group,income_level,income_minmax
0,101,25.0,30000,18000,Delhi,2,Regular,12000,40.000000,Young Adult,Medium,0.428571
1,102,42.0,50000,32000,Mumbai,5,Premium,18000,36.000000,Middle-aged,High,0.714286
2,103,17.0,18000,12000,Delhi,1,New,6000,33.333333,Minor,Low,0.257143
3,104,60.0,70000,45000,Pune,8,Premium,25000,35.714286,Senior,High,1.000000
4,105,35.0,40000,25000,Mumbai,4,Regular,15000,37.500000,Young Adult,Medium,0.571429
5,106,NaN,0,10000,Pune,0,New,-10000,NaN,Invalid/Missing,Low,0.000000
6,107,150.0,55000,30000,Delhi,6,Premium,25000,45.454545,Invalid/Missing,High,0.785714
7,108,29.0,35000,20000,Mumbai,3,Regular,15000,42.857143,Young Adult,Medium,0.500000


#### Q5. Standardize a Numerical Feature
Create an expenses_zscore column by standardizing monthly expenses using the mean and population standard deviation (ddof=0). Explain how you would handle a zero standard deviation.

In [24]:
df['expense_z_score']=(df['monthly_expenses']-df['monthly_expenses'].mean())/df['monthly_expenses'].std()
df

,customer_id,age,monthly_income,monthly_expenses,city,purchases,customer_type,monthly_savings,savings_rate,age_group,income_level,income_minmax,expense_z_score
0,101,25.0,30000,18000,Delhi,2,Regular,12000,40.000000,Young Adult,Medium,0.428571,-0.519430
1,102,42.0,50000,32000,Mumbai,5,Premium,18000,36.000000,Middle-aged,High,0.714286,0.692573
2,103,17.0,18000,12000,Delhi,1,New,6000,33.333333,Minor,Low,0.257143,-1.038860
3,104,60.0,70000,45000,Pune,8,Premium,25000,35.714286,Senior,High,1.000000,1.818004
4,105,35.0,40000,25000,Mumbai,4,Regular,15000,37.500000,Young Adult,Medium,0.571429,0.086572
5,106,NaN,0,10000,Pune,0,New,-10000,NaN,Invalid/Missing,Low,0.000000,-1.212003
6,107,150.0,55000,30000,Delhi,6,Premium,25000,45.454545,Invalid/Missing,High,0.785714,0.519430
7,108,29.0,35000,20000,Mumbai,3,Regular,15000,42.857143,Young Adult,Medium,0.500000,-0.346287


#### Q6. Encode Categorical Features
Use one-hot encoding to convert city into indicator columns. Keep the original DataFrame unchanged and explain what drop_first=True does.

In [29]:
df1=pd.get_dummies(df,columns=['city'],
                  drop_first=True,
                  dtype=int)
df1

,customer_id,age,monthly_income,monthly_expenses,purchases,customer_type,monthly_savings,savings_rate,age_group,income_level,income_minmax,expense_z_score,city_Mumbai,city_Pune
0,101,25.0,30000,18000,2,Regular,12000,40.000000,Young Adult,Medium,0.428571,-0.519430,0,0
1,102,42.0,50000,32000,5,Premium,18000,36.000000,Middle-aged,High,0.714286,0.692573,1,0
2,103,17.0,18000,12000,1,New,6000,33.333333,Minor,Low,0.257143,-1.038860,0,0
3,104,60.0,70000,45000,8,Premium,25000,35.714286,Senior,High,1.000000,1.818004,0,1
4,105,35.0,40000,25000,4,Regular,15000,37.500000,Young Adult,Medium,0.571429,0.086572,1,0
5,106,NaN,0,10000,0,New,-10000,NaN,Invalid/Missing,Low,0.000000,-1.212003,0,1
6,107,150.0,55000,30000,6,Premium,25000,45.454545,Invalid/Missing,High,0.785714,0.519430,0,0
7,108,29.0,35000,20000,3,Regular,15000,42.857143,Young Adult,Medium,0.500000,-0.346287,1,0


#### What does drop_first=True do?
It drops the first category's indicator column, leaving \(k-1\) columns for \(k\) categories.

This can help avoid perfect multicollinearity when using an intercept-based linear regression model.

#### Q7. Map Categories to Numbers
Create a customer_type_code column by mapping New to 0, Regular to 1, and Premium to 2. Ensure that unexpected or unmapped categories do not silently become valid codes.

In [37]:
mapping={
    'New':0,
    'Regular':1,
    'Premium':2
}
df1['customer_type_code']=df1['customer_type'].map(mapping)

invalid_mask = (
    df1['customer_type'].notna()
    & df1['customer_type_code'].isna()
)

if invalid_mask.any():
    print("Unexpected categories:")
    print(df.loc[invalid_mask, 'customer_type'].unique())
df1

,customer_id,age,monthly_income,monthly_expenses,purchases,customer_type,monthly_savings,savings_rate,age_group,income_level,income_minmax,expense_z_score,city_Mumbai,city_Pune,customer_type_code
0,101,25.0,30000,18000,2,Regular,12000,40.000000,Young Adult,Medium,0.428571,-0.519430,0,0,1
1,102,42.0,50000,32000,5,Premium,18000,36.000000,Middle-aged,High,0.714286,0.692573,1,0,2
2,103,17.0,18000,12000,1,New,6000,33.333333,Minor,Low,0.257143,-1.038860,0,0,0
3,104,60.0,70000,45000,8,Premium,25000,35.714286,Senior,High,1.000000,1.818004,0,1,2
4,105,35.0,40000,25000,4,Regular,15000,37.500000,Young Adult,Medium,0.571429,0.086572,1,0,1
5,106,NaN,0,10000,0,New,-10000,NaN,Invalid/Missing,Low,0.000000,-1.212003,0,1,0
6,107,150.0,55000,30000,6,Premium,25000,45.454545,Invalid/Missing,High,0.785714,0.519430,0,0,2
7,108,29.0,35000,20000,3,Regular,15000,42.857143,Young Adult,Medium,0.500000,-0.346287,1,0,1


#### Q8. Create Interaction Features
Create an income_per_purchase column by dividing monthly income by the number of purchases. Avoid infinite values or misleading results when purchases equal zero. Explain what the result should represent for a customer with zero purchases.

In [50]:
df1['income_per_purchase']=np.where(
    df1['monthly_income']>0,
    df1['purchases']/df1['monthly_income'].replace(0,np.nan),np.nan)
df1


,customer_id,age,monthly_income,monthly_expenses,purchases,customer_type,monthly_savings,savings_rate,age_group,income_level,income_minmax,expense_z_score,city_Mumbai,city_Pune,customer_type_code,income_per_purchase
0,101,25.0,30000,18000,2,Regular,12000,40.000000,Young Adult,Medium,0.428571,-0.519430,0,0,1,0.000067
1,102,42.0,50000,32000,5,Premium,18000,36.000000,Middle-aged,High,0.714286,0.692573,1,0,2,0.000100
2,103,17.0,18000,12000,1,New,6000,33.333333,Minor,Low,0.257143,-1.038860,0,0,0,0.000056
3,104,60.0,70000,45000,8,Premium,25000,35.714286,Senior,High,1.000000,1.818004,0,1,2,0.000114
4,105,35.0,40000,25000,4,Regular,15000,37.500000,Young Adult,Medium,0.571429,0.086572,1,0,1,0.000100
5,106,NaN,0,10000,0,New,-10000,NaN,Invalid/Missing,Low,0.000000,-1.212003,0,1,0,NaN
6,107,150.0,55000,30000,6,Premium,25000,45.454545,Invalid/Missing,High,0.785714,0.519430,0,0,2,0.000109
7,108,29.0,35000,20000,3,Regular,15000,42.857143,Young Adult,Medium,0.500000,-0.346287,1,0,1,0.000086


#### Q9. Build Multiple Business Features
Create the following three columns:
- expense_ratio: monthly expenses divided by monthly income, expressed as a percentage.
- high_spender: True if monthly expenses exceed 30,000; otherwise False.
- purchase_segment: No Purchases for 0 purchases, Low for 1–2, Medium for 3–5, and High for 6 or more.
Handle division by zero safely.

In [56]:
df1['expense_ratio']=np.where(
    df1['monthly_income']>0,
    (df1['monthly_expenses']/df['monthly_income'])*100,np.nan)
df1

,customer_id,age,monthly_income,monthly_expenses,purchases,customer_type,monthly_savings,savings_rate,age_group,income_level,income_minmax,expense_z_score,city_Mumbai,city_Pune,customer_type_code,income_per_purchase,expense_ratio
0,101,25.0,30000,18000,2,Regular,12000,40.000000,Young Adult,Medium,0.428571,-0.519430,0,0,1,0.000067,60.000000
1,102,42.0,50000,32000,5,Premium,18000,36.000000,Middle-aged,High,0.714286,0.692573,1,0,2,0.000100,64.000000
2,103,17.0,18000,12000,1,New,6000,33.333333,Minor,Low,0.257143,-1.038860,0,0,0,0.000056,66.666667
3,104,60.0,70000,45000,8,Premium,25000,35.714286,Senior,High,1.000000,1.818004,0,1,2,0.000114,64.285714
4,105,35.0,40000,25000,4,Regular,15000,37.500000,Young Adult,Medium,0.571429,0.086572,1,0,1,0.000100,62.500000
5,106,NaN,0,10000,0,New,-10000,NaN,Invalid/Missing,Low,0.000000,-1.212003,0,1,0,NaN,NaN
6,107,150.0,55000,30000,6,Premium,25000,45.454545,Invalid/Missing,High,0.785714,0.519430,0,0,2,0.000109,54.545455
7,108,29.0,35000,20000,3,Regular,15000,42.857143,Young Adult,Medium,0.500000,-0.346287,1,0,1,0.000086,57.142857


#### Q10. Mini-Project: Build a Feature Engineering Pipeline
Create a new DataFrame without modifying the original df.
1. Replace invalid ages outside the range 0–120 with missing values.
2. Create monthly_savings.
3. Create savings_rate, safely handling zero income.
4. Create age_group.
5. Create income_level.
6. Create high_spender.
7. Verify that the output contains all expected columns, has no infinite values in engineered numerical features, and retains the same number of rows as the original DataFrame.

In [61]:
df = pd.DataFrame({
    'customer_id': [101, 102, 103, 104, 105, 106, 107, 108],
    'age': [25, 42, 17, 60, 35, np.nan, 150, 29],
    'monthly_income': [30000, 50000, 18000, 70000, 40000, 0, 55000, 35000],
    'monthly_expenses': [18000, 32000, 12000, 45000, 25000, 10000, 30000, 20000],
    'city': ['Delhi', 'Mumbai', 'Delhi', 'Pune', 'Mumbai', 'Pune', 'Delhi', 'Mumbai'],
    'purchases': [2, 5, 1, 8, 4, 0, 6, 3],
    'customer_type': ['Regular', 'Premium', 'New', 'Premium', 'Regular', 'New', 'Premium', 'Regular']
})
dff=df.copy()
dff

,customer_id,age,monthly_income,monthly_expenses,city,purchases,customer_type
0,101,25.0,30000,18000,Delhi,2,Regular
1,102,42.0,50000,32000,Mumbai,5,Premium
2,103,17.0,18000,12000,Delhi,1,New
3,104,60.0,70000,45000,Pune,8,Premium
4,105,35.0,40000,25000,Mumbai,4,Regular
5,106,NaN,0,10000,Pune,0,New
6,107,150.0,55000,30000,Delhi,6,Premium
7,108,29.0,35000,20000,Mumbai,3,Regular


In [65]:
#Replace invalid ages outside the range 0–120 with missing values.
dff.loc[~df['age'].between(0,120),'age']=np.nan
dff

,customer_id,age,monthly_income,monthly_expenses,city,purchases,customer_type
0,101,25.0,30000,18000,Delhi,2,Regular
1,102,42.0,50000,32000,Mumbai,5,Premium
2,103,17.0,18000,12000,Delhi,1,New
3,104,60.0,70000,45000,Pune,8,Premium
4,105,35.0,40000,25000,Mumbai,4,Regular
5,106,NaN,0,10000,Pune,0,New
6,107,NaN,55000,30000,Delhi,6,Premium
7,108,29.0,35000,20000,Mumbai,3,Regular


In [67]:
#Create monthly_savings.
dff['monthly_savings']=dff['monthly_income']-dff['monthly_expenses']
dff

,customer_id,age,monthly_income,monthly_expenses,city,purchases,customer_type,monthly_savings
0,101,25.0,30000,18000,Delhi,2,Regular,12000
1,102,42.0,50000,32000,Mumbai,5,Premium,18000
2,103,17.0,18000,12000,Delhi,1,New,6000
3,104,60.0,70000,45000,Pune,8,Premium,25000
4,105,35.0,40000,25000,Mumbai,4,Regular,15000
5,106,NaN,0,10000,Pune,0,New,-10000
6,107,NaN,55000,30000,Delhi,6,Premium,25000
7,108,29.0,35000,20000,Mumbai,3,Regular,15000


In [69]:
#Create savings_rate, safely handling zero income.
dff['saving_rate']=np.where(
    dff['monthly_income']>0,
    (dff['monthly_savings']/dff['monthly_income'])*100,np.nan)
dff

,customer_id,age,monthly_income,monthly_expenses,city,purchases,customer_type,monthly_savings,saving_rate
0,101,25.0,30000,18000,Delhi,2,Regular,12000,40.000000
1,102,42.0,50000,32000,Mumbai,5,Premium,18000,36.000000
2,103,17.0,18000,12000,Delhi,1,New,6000,33.333333
3,104,60.0,70000,45000,Pune,8,Premium,25000,35.714286
4,105,35.0,40000,25000,Mumbai,4,Regular,15000,37.500000
5,106,NaN,0,10000,Pune,0,New,-10000,NaN
6,107,NaN,55000,30000,Delhi,6,Premium,25000,45.454545
7,108,29.0,35000,20000,Mumbai,3,Regular,15000,42.857143


In [77]:
#Create age_group.
conditions=[
    dff['age']<18,
    dff['age'].between(18,40),
    dff['age']>40
]
choices=['Young','Adult','Mature']
dff['age']=np.select(conditions,choices,default='Not Mentioned')
dff

,customer_id,age,monthly_income,monthly_expenses,city,purchases,customer_type,monthly_savings,saving_rate
0,101,Adult,30000,18000,Delhi,2,Regular,12000,40.000000
1,102,Mature,50000,32000,Mumbai,5,Premium,18000,36.000000
2,103,Young,18000,12000,Delhi,1,New,6000,33.333333
3,104,Mature,70000,45000,Pune,8,Premium,25000,35.714286
4,105,Adult,40000,25000,Mumbai,4,Regular,15000,37.500000
5,106,Not Mentioned,0,10000,Pune,0,New,-10000,NaN
6,107,Not Mentioned,55000,30000,Delhi,6,Premium,25000,45.454545
7,108,Adult,35000,20000,Mumbai,3,Regular,15000,42.857143


In [81]:
#Create income_level.
conditions=[
    dff['monthly_income']<=20000,
    dff['monthly_income'].between(20001,50000),
    dff['monthly_income']>50000
]
labels=['Low','Medium','High']
dff['income_level']=np.select(conditions,labels,default='Not mentioned')
dff

,customer_id,age,monthly_income,monthly_expenses,city,purchases,customer_type,monthly_savings,saving_rate,income_level
0,101,Adult,30000,18000,Delhi,2,Regular,12000,40.000000,Medium
1,102,Mature,50000,32000,Mumbai,5,Premium,18000,36.000000,Medium
2,103,Young,18000,12000,Delhi,1,New,6000,33.333333,Low
3,104,Mature,70000,45000,Pune,8,Premium,25000,35.714286,High
4,105,Adult,40000,25000,Mumbai,4,Regular,15000,37.500000,Medium
5,106,Not Mentioned,0,10000,Pune,0,New,-10000,NaN,Low
6,107,Not Mentioned,55000,30000,Delhi,6,Premium,25000,45.454545,High
7,108,Adult,35000,20000,Mumbai,3,Regular,15000,42.857143,Medium


In [83]:
#Create high_spender.
dff['high_spender']=np.where(
    df['monthly_expenses']>24000,True,False)
dff

,customer_id,age,monthly_income,monthly_expenses,city,purchases,customer_type,monthly_savings,saving_rate,income_level,high_spender
0,101,Adult,30000,18000,Delhi,2,Regular,12000,40.000000,Medium,False
1,102,Mature,50000,32000,Mumbai,5,Premium,18000,36.000000,Medium,True
2,103,Young,18000,12000,Delhi,1,New,6000,33.333333,Low,False
3,104,Mature,70000,45000,Pune,8,Premium,25000,35.714286,High,True
4,105,Adult,40000,25000,Mumbai,4,Regular,15000,37.500000,Medium,True
5,106,Not Mentioned,0,10000,Pune,0,New,-10000,NaN,Low,False
6,107,Not Mentioned,55000,30000,Delhi,6,Premium,25000,45.454545,High,True
7,108,Adult,35000,20000,Mumbai,3,Regular,15000,42.857143,Medium,False
